# PR to PO Test Notebook

**Purpose:** Demo scope — **document upload** and **document categorization** only. This notebook ingests PR attachments from a folder, lists them, and classifies each file (Quotation, Contract, SOW, Invoice, etc.) using the LLM. No parsing, database, or compliance checks in this notebook.

**Environment:** Run locally with **Groq** (`GROQ_API_KEY` in `.env`) or in production with AWS Bedrock. Run Section 0 → 1 → 2 → 3 → 4 in order.

**Getting started**

1. Set **GROQ_API_KEY** in `.env` (or AWS vars for production).
2. Run **Section 0** (env detection) → **Section 1** (paths + LLM) → **Section 2** (schemas) → **Section 3** (list documents) → **Section 4** (document classification).
3. **Output:** `uploaded` (list of files) and `classified_documents` (document type + confidence per file). Nothing is written to a database or to the output folder in this notebook.

**What this notebook implements so far:**  
- **§0:** Environment detection (local vs AWS).  
- **§1:** Paths (input/output/schemas) and LLM (Groq or Bedrock).  
- **§2:** Load schema references (for classification prompt; full pipeline uses these for parsing later).  
- **§3:** List uploaded documents (from input folder or synthetic_data).  
- **§4:** Document categorization — each attachment classified by the LLM (Quotation, Contract, SOW, etc.).  

**Not in this notebook:** Document parsing (Quote/MSA extraction), database persistence, compliance checks, or writing to the output folder.

---
## 0. Environment detection

**What this does:** Reads environment variables to choose local (Groq) or production (AWS Bedrock). Sets `ENV_MODE`, `USE_GROQ`, `USE_BEDROCK`, and `DATABASE_URL` for later cells.

In [1]:
import os
import json

def detect_env():
    use_aws = (
        os.environ.get("AWS_REGION") or os.environ.get("AWS_PROFILE")
    ) and os.environ.get("BEDROCK_MODEL_ID")
    use_groq = bool(os.environ.get("GROQ_API_KEY"))
    
    if use_aws:
        return "aws"
    if use_groq:
        return "local"
    return "local"  # default to local for notebook

ENV_MODE = detect_env()
print(f"Detected mode: {ENV_MODE}")

# LLM: Bedrock (AWS) or Groq (local)
USE_BEDROCK = ENV_MODE == "aws"
USE_GROQ = ENV_MODE == "local"

# Embeddings: Titan (AWS) or sentence-transformers (local)
USE_TITAN = ENV_MODE == "aws"
USE_LOCAL_EMBEDDINGS = ENV_MODE == "local"

# DB: RDS (AWS) or SQLite/Postgres (local)
DATABASE_URL = os.environ.get("DATABASE_URL") or os.environ.get("LOCAL_DB_PATH") or "sqlite:///pr_validation.db"
print(f"Database: {DATABASE_URL}")

Detected mode: local
Database: sqlite:///pr_validation.db


---
## 1. Imports and config

**What this does:** Loads `.env`, sets **BASE_DIR** / **SCHEMAS_DIR** / **INPUT_DIR** / **OUTPUT_DIR**, creates input/output folders if missing, and creates the **LLM** (Groq or Bedrock) used for document classification.

In [2]:
# Optional: load .env if present
try:
    from dotenv import load_dotenv
    load_dotenv()
except ImportError:
    pass

import os
from pathlib import Path

# Base paths: run from repo root or from document_processing_rag
CWD = Path(os.getcwd())
if (CWD / "schemas").exists():
    BASE_DIR = CWD
elif (CWD / "document_processing_rag" / "schemas").exists():
    BASE_DIR = CWD / "document_processing_rag"
else:
    BASE_DIR = CWD

SCHEMAS_DIR = BASE_DIR / "schemas"
INPUT_DIR = BASE_DIR / "input"
OUTPUT_DIR = BASE_DIR / "output"
INPUT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
# Fallback if input is empty: synthetic_data/multi item example/live
if (BASE_DIR.parent / "synthetic_data").exists():
    SYNTHETIC_DIR = BASE_DIR.parent / "synthetic_data" / "multi item example" / "live"
else:
    SYNTHETIC_DIR = BASE_DIR / "synthetic_data" / "multi item example" / "live"

print("BASE_DIR:", BASE_DIR)
print("SCHEMAS_DIR:", SCHEMAS_DIR, "exists:", SCHEMAS_DIR.exists())
print("INPUT_DIR:", INPUT_DIR, "exists:", INPUT_DIR.exists())
print("OUTPUT_DIR:", OUTPUT_DIR, "exists:", OUTPUT_DIR.exists())

BASE_DIR: m:\AI_consulting\2025\Bhavin\JSONs\PRtoPOAgent\document_processing_rag
SCHEMAS_DIR: m:\AI_consulting\2025\Bhavin\JSONs\PRtoPOAgent\document_processing_rag\schemas exists: True
INPUT_DIR: m:\AI_consulting\2025\Bhavin\JSONs\PRtoPOAgent\document_processing_rag\input exists: True
OUTPUT_DIR: m:\AI_consulting\2025\Bhavin\JSONs\PRtoPOAgent\document_processing_rag\output exists: True


In [3]:
# LangChain and LLM
try:
    from langchain_core.messages import HumanMessage, SystemMessage
    from langchain_core.output_parsers import JsonOutputParser
except ImportError:
    raise ImportError("Install: pip install langchain-core")

if USE_GROQ:
    try:
        from langchain_groq import ChatGroq
        llm = ChatGroq(
            model=os.environ.get("LOCAL_LLM_MODEL", "llama-3.1-8b-instant"),
            api_key=os.environ.get("GROQ_API_KEY"),
            temperature=0.1,
        )
        print("LLM: Groq (local)")
    except ImportError:
        raise ImportError("Install: pip install langchain-groq")
elif USE_BEDROCK:
    try:
        from langchain_aws import ChatBedrock
        llm = ChatBedrock(
            model_id=os.environ.get("BEDROCK_MODEL_ID", "anthropic.claude-3-5-haiku-20241022-v2:0"),
            region_name=os.environ.get("AWS_REGION", "us-east-1"),
            temperature=0.1,
        )
        print("LLM: Bedrock (AWS)")
    except ImportError:
        raise ImportError("Install: pip install langchain-aws boto3")
else:
    raise RuntimeError("Set GROQ_API_KEY (local) or AWS + BEDROCK_MODEL_ID (production)")

LLM: Groq (local)


---
## 2. Load schema references

**What this does:** Loads schema JSONs from `schemas/`, including the single **PR template** (`pr.json`: header + attachments + line_items). Used for classification prompt structure; full pipeline uses these for parsing and checks later.

In [4]:
def load_schema(name: str) -> dict:
    path = SCHEMAS_DIR / f"{name}.json"
    if not path.exists():
        raise FileNotFoundError(str(path))
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

def load_check_result(check_num: int) -> dict:
    if check_num == 1:
        path = SCHEMAS_DIR / "compliance_checks" / "check_01_attachment_existence_classification" / "check_01_result.json"
    elif check_num == 2:
        path = SCHEMAS_DIR / "compliance_checks" / "check_02_document_validity_applicability" / "check_02_result.json"
    else:
        raise ValueError(f"Check {check_num} schema not yet added")
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

# Load extraction schemas (for parsing in full pipeline). PR is single combined template: header + attachments + line_items.
schemas = {}
for name in ["pr", "quote", "msa"]:
    try:
        schemas[name] = load_schema(name)
        print(f"Loaded schema: {name}")
    except FileNotFoundError as e:
        print(f"Skip {name}: {e}")

# Check result schemas (for full pipeline)
check1_schema = load_check_result(1)
check2_schema = load_check_result(2)
print("Check 1 & 2 result schemas loaded.")

Loaded schema: pr
Loaded schema: quote
Loaded schema: msa
Check 1 & 2 result schemas loaded.


---
## 3. Document upload

**What this does:** Lists documents in the input folder (or synthetic_data) — PDF, Word, Excel. **Output:** `uploaded` (list of path, filename, size_bytes).

In [5]:
def list_uploaded_documents(source_dir: Path = None) -> list[dict]:
    """List documents available for processing. Each item: {path, filename, size_bytes}."""
    source_dir = source_dir or INPUT_DIR
    if not source_dir.exists():
        source_dir = SYNTHETIC_DIR
    if not source_dir.exists():
        return []
    out = []
    for f in source_dir.iterdir():
        if f.is_file() and f.suffix.lower() in (".pdf", ".docx", ".xlsx", ".doc", ".xls"):
            out.append({
                "path": str(f),
                "filename": f.name,
                "size_bytes": f.stat().st_size,
            })
    return sorted(out, key=lambda x: x["filename"])

uploaded = list_uploaded_documents()
for u in uploaded:
    print(u["filename"], "(", u["size_bytes"], "bytes )")
if not uploaded:
    print("No documents found. Place PDF/Word/Excel in the input folder (document_processing_rag/input) or in synthetic_data/.../live.")

01_PR_Header_and_LineItems.xlsx ( 9436 bytes )
02_CDW_Quote_Q25-0847.pdf ( 6118 bytes )
02b_MSA_CDW_2024_0156_Executed.pdf ( 8365 bytes )


---
## 4. Document categorization

**What this does:** Classifies **each uploaded attachment** into one document type: **PR Form** (Excel with PR Header/Line Items/Attachments — one or three sheets), Quotation, Contract, SOW, Service Agreement, Invoice, BidSummary, Justification, Spec, Other. Uses sheet names and column headers so PR form Excel is not miscategorized. **Output:** `classified_documents` — one dict per file with `filename`, `document_type`, `confidence`, `reason`. This is the gatekeeper step; the full pipeline uses this to route Quote vs MSA parsing and for compliance checks.

In [6]:
def extract_text_from_file(filepath: str, max_chars: int = 2000) -> str:
    """Extract first max_chars of text for classification hint. Excel: sheet names + headers + first rows (supports single-sheet or multi-sheet PR Header/Line Items/Attachments)."""
    path = Path(filepath)
    suffix = path.suffix.lower()
    try:
        if suffix == ".pdf":
            try:
                import pypdf
                reader = pypdf.PdfReader(filepath)
                text = ""
                for i, page in enumerate(reader.pages):
                    if i >= 3:
                        break
                    text += page.extract_text() or ""
                return (text or path.name)[:max_chars]
            except ImportError:
                return path.name
        if suffix in (".docx", ".doc"):
            try:
                import docx
                doc = docx.Document(filepath)
                return "\n".join(p.text for p in doc.paragraphs[:30])[:max_chars]
            except ImportError:
                return path.name
        if suffix == ".xlsx":
            try:
                from openpyxl import load_workbook
                wb = load_workbook(filepath, read_only=False, data_only=True)
                parts = [f"Sheets: {', '.join(wb.sheetnames)}"]
                for sheet_name in wb.sheetnames[:5]:
                    ws = wb[sheet_name]
                    rows = list(ws.iter_rows(max_row=4, values_only=True))
                    if rows:
                        header = " | ".join(str(c) if c is not None else "" for c in rows[0][:20])
                        parts.append(f"[{sheet_name}] Headers: {header}")
                        for r in rows[1:4]:
                            parts.append(" | ".join(str(c) if c is not None else "" for c in r[:20]))
                return ("\n".join(parts) or path.name)[:max_chars]
            except Exception:
                return path.name
        if suffix == ".xls":
            return path.name
    except Exception as e:
        return path.name + " " + str(e)[:200]
    return path.name

# Classification system prompt includes JSON example. PR Form = PR data (Excel).
CLASSIFY_EXAMPLE = json.dumps({"document_type": "PR Form", "confidence": 0.95, "reason": "Excel with sheets PR Header, PR Line Items, PR Attachments; columns PR Number, PR Status, Total Estimated Value."})
CLASSIFY_SYSTEM = """You are a procurement document classifier. Given the file name and a short excerpt, classify into exactly one type: PR Form, Quotation, Contract, SOW, Service Agreement, Invoice, BidSummary, Justification, Spec, Other.
PR Form: Excel or workbook with sheet(s) named 'PR Header', 'PR Line Items', 'PR Attachments' OR columns/headers like PR Number, PR Status, Requestor Name, Total Estimated Value, Line Items, Ship-To Address. This is the purchase requisition form data (header, line items, attachments). Can be one sheet with all columns or three separate sheets. NOT a quotation.
Quotation: Quote, Quotation, Valid Until, line items with prices, supplier letterhead.
Contract: Agreement, Contract, Master Service Agreement, parties, effective/expiry.
SOW: Statement of Work, Scope of Work, deliverables, milestones.
Invoice: Invoice, Bill To, Payment Due.
Respond with JSON only. Example output: """ + CLASSIFY_EXAMPLE

def classify_document(llm, filepath: str, filename: str, excerpt: str = None) -> dict:
    excerpt = excerpt or extract_text_from_file(filepath)
    prompt = f"File: {filename}\n\nExcerpt:\n{excerpt[:1500]}\n\nClassify. JSON only."
    msg = [SystemMessage(content=CLASSIFY_SYSTEM), HumanMessage(content=prompt)]
    out = llm.invoke(msg)
    text = out.content if hasattr(out, "content") else str(out)
    try:
        if "```" in text:
            text = text.split("```")[1].replace("json", "").strip()
        return json.loads(text)
    except json.JSONDecodeError:
        return {"document_type": "Other", "confidence": 0.5, "reason": "Parse failed"}

# Run classification on uploaded docs
classified_documents = []
for u in uploaded:
    result = classify_document(llm, u["path"], u["filename"])
    result["filename"] = u["filename"]
    classified_documents.append(result)
    print(u["filename"], "->", result.get("document_type"), "(", result.get("confidence"), ")")

01_PR_Header_and_LineItems.xlsx -> PR Form ( 0.99 )
02_CDW_Quote_Q25-0847.pdf -> Quotation ( 0.95 )
02b_MSA_CDW_2024_0156_Executed.pdf -> Contract ( 0.99 )
